# 2-05 공시 본문 — 원문에서 숫자·핵심 뽑아내기

**질문 예시:** "오리온 실적 공시 본문에서 국가별 순매출 뽑아줘", "오늘 공매도 과열 지정 공시들 대상종목·ISIN 정리해줘", "오늘 나온 잠정실적 공시 매출·영업이익 표로 모아줘"

DeepSearch 공시 검색은 **제목·시각·종목·원문 링크**까지만 줍니다(본문 필드 `content`에는 제목만 들어 있음).
그래서 이 노트북은 DeepSearch로 공시를 **찾고**, 링크가 가리키는 **KIND(거래소)·DART(금감원) 공개 뷰어 페이지**에서 본문을 직접 받아 표·숫자를 뽑습니다.
API 키가 필요한 OpenDART는 쓰지 않습니다.

| 링크 | 본문을 받는 경로 (2026-11-25 확인) |
|---|---|
| `kind.krx.co.kr/common/disclsviewer.do?method=search&acptno=…` | 뷰어 페이지의 문서번호 → `method=searchContents&docNo=…` → 실제 본문 `.htm` |
| `dart.fss.or.kr/dsaf001/main.do?rcpNo=…` | 페이지 안 `viewDoc("rcpNo","dcmNo",…)` 인자 → `report/viewer.do?…` 본문 HTML |

CHECK 원본(`2_공시뉴스/05_공시본문`)의 두 예시(오리온 국가별 매출, 공매도 과열 공시 구조화)를 그대로 재현하고, 잠정실적 일괄 추출을 더했습니다.

[![Colab에서 열기](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/beaten-by-the-market/deepsearch/blob/main/scenarios/2_disclosure_news/05_disclosure_text.ipynb)

## 0. 준비
Colab 왼쪽 🔑(Secrets)에 `DEEPSEARCH_API_KEY`를 등록하고 이 노트북의 액세스를 켠 뒤 아래 두 셀을 실행하세요. 두 번째 셀은 모든 시나리오에 공통인 헬퍼입니다(접어 두어도 됩니다).

In [1]:
# Colab에서만: 한글 차트 폰트 설치 (로컬에서는 건너뜀)
import sys
if "google.colab" in sys.modules:
    !pip -q install koreanize-matplotlib

In [2]:
# ── DeepSearch 헬퍼 (모든 시나리오 노트북 공통) ─────────────────────────────
# Colab: 왼쪽 🔑(Secrets)에 이름 DEEPSEARCH_API_KEY 로 키를 넣고 "노트북 액세스"를 켜세요.
#        키는 "아이디:비밀번호" 형식이든 base64 문자열이든 됩니다. 키 값은 출력하지 않습니다.
# 로컬: 환경변수 DEEPSEARCH_API_KEY / API_KEY 또는 상위 폴더의 .env 를 읽습니다.
import base64, json, os, sys, time
from datetime import date, datetime, timedelta

import pandas as pd
import requests

IN_COLAB = "google.colab" in sys.modules
API_URL = "https://api.deepsearch.com/v1/compute"

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 60)


def _norm_key(k):
    k = k.strip().strip('"').strip("'")
    return base64.b64encode(k.encode()).decode() if ":" in k else k


def _find_key():
    if IN_COLAB:
        try:
            from google.colab import userdata
            k = userdata.get("DEEPSEARCH_API_KEY")
            if k:
                return _norm_key(k)
        except Exception:
            pass
    for name in ("DEEPSEARCH_API_KEY", "API_KEY"):
        if os.environ.get(name):
            return _norm_key(os.environ[name])
    d = os.path.abspath(os.getcwd())
    while True:
        p = os.path.join(d, ".env")
        if os.path.isfile(p):
            for line in open(p, encoding="utf-8-sig"):
                if "=" in line and not line.strip().startswith("#"):
                    k, v = line.split("=", 1)
                    if k.strip() in ("DEEPSEARCH_API_KEY", "API_KEY") and v.strip():
                        return _norm_key(v)
        if os.path.dirname(d) == d:
            break
        d = os.path.dirname(d)
    from getpass import getpass
    return _norm_key(getpass("DeepSearch API 키: "))


_KEY = _find_key()


class DSError(Exception):
    pass


def ds(query, timeout=180, retries=3):
    """compute 호출 → 응답 JSON. 서버가 돌려준 오류(data.exceptions)는 DSError."""
    last = None
    for i in range(retries):
        try:
            r = requests.post(API_URL, json={"input": query}, timeout=timeout,
                              headers={"Authorization": "Basic " + _KEY})
            body = r.json()
        except Exception as e:  # 네트워크·일시 오류만 재시도
            last = e
            time.sleep(3 * (i + 1))
            continue
        exc = (body.get("data") or {}).get("exceptions") or []
        if exc or r.status_code >= 400:
            msg = "; ".join(f"{x.get('class', '')}: {' '.join(map(str, x.get('args') or [x.get('message', '')]))}"
                            if isinstance(x, dict) else str(x) for x in exc)
            raise DSError(f"{r.status_code} {msg or body}"[:500])
        return body
    raise DSError(f"요청 실패: {last}")


def _pod(body):
    pods = (body.get("data") or {}).get("pods") or []
    return pods[1] if len(pods) >= 2 else {}


def ds_df(query, **kw):
    """DataFrame 결과(재무·시세·경제지표 등) → pandas DataFrame. 결과가 없으면 빈 DataFrame."""
    c = _pod(ds(query, **kw)).get("content") or {}
    data = c.get("data") if isinstance(c, dict) else None
    if not isinstance(data, dict) or not data:
        return pd.DataFrame()
    cols = [x for x in (c.get("columns") or data.keys()) if x in data]
    df = pd.DataFrame({x: data[x] for x in cols})
    for x in ("date", "created_at"):
        if x in df and df[x].dtype == object:
            df[x] = pd.to_datetime(df[x].astype(str).str[:19], errors="coerce")
    return df


def _polarity(x):
    """문서 긍부정: polarity.name, 없으면 customized_ml.kis 모델 값(최근 뉴스는 절반가량이 이쪽에만 있음)."""
    p = x.get("polarity")
    if isinstance(p, dict) and p.get("name"):
        return p["name"]
    k = (((x.get("customized_ml") or {}).get("kis") or {}).get("ip_multitask") or {}).get("polarity") or {}
    return k.get("name")


def doc_search(category, section, query, date_from, date_to, count=100, pages=1):
    """DocumentSearch → (총건수, 문서 DataFrame). category: news/research/company/patent. 기간은 1년 이내.
    종목 지정은 securities.symbol:005930 (접두어 없는 6자리)이 뉴스·리서치·공시 모두에서 가장 정확합니다.
    securities.symbol:KRX:005930 은 키워드 검색으로 바뀌어 건수가 줄고, securities.name: 은 리서치(공식명
    '에스케이하이닉스')·공시(DART명 '현대자동차')에서 놓칩니다(2026-11-25 확인). 여러 종목: securities.symbol:(005930 or 000660)"""
    rows, total = [], 0
    for page in range(1, pages + 1):
        q = (f'DocumentSearch(["{category}"],{json.dumps(section, ensure_ascii=False)},'
             f'{json.dumps(query, ensure_ascii=False)},count={count},page={page},'
             f'date_from={date_from},date_to={date_to})')
        c = _pod(ds(q)).get("content") or {}
        d = c.get("data") if isinstance(c.get("data"), dict) else c
        total = d.get("total_matches") or 0
        docs = d.get("docs") or []
        for x in docs:
            secs = x.get("securities") or []
            rows.append({
                "created_at": pd.to_datetime(str(x.get("created_at"))[:19], errors="coerce"),
                "publisher": x.get("publisher"),
                "title": " ".join(str(x.get("title") or "").split()),
                "stocks": ", ".join(s.get("name", "") for s in secs),
                "symbols": ", ".join(s.get("symbol", "") for s in secs),
                "polarity": _polarity(x),
                "url": x.get("content_url"),
            })
        if len(docs) < count:
            break
    cols = ["created_at", "publisher", "title", "stocks", "symbols", "polarity", "url"]
    return total, pd.DataFrame(rows, columns=cols)


def doc_trends(category, section, query, date_from, date_to, interval="1d"):
    """DocumentTrends → DataFrame(date, count)."""
    q = (f'DocumentTrends(["{category}"],{json.dumps(section, ensure_ascii=False)},'
         f'{json.dumps(query, ensure_ascii=False)},interval="{interval}",'
         f'date_from={date_from},date_to={date_to})')
    c = _pod(ds(q)).get("content") or {}
    d = c.get("data") if isinstance(c.get("data"), dict) else c
    trends = d.get("trends") or []
    b = trends[0].get("buckets", []) if trends else []
    df = pd.DataFrame([{"date": x.get("key_as_string") or x.get("key"), "count": x.get("count", 0)} for x in b])
    if not df.empty:
        df["date"] = pd.to_datetime(df["date"].astype(str).str[:19], errors="coerce")
    return df


def stock_prices(entities, date_from, date_to, columns=None):
    """종목 목록의 일별 시세. entities 예: ["KRX:005930", "KRX:000660"] 또는 "KRX:005930".
    가격은 수정주가, 거래량·거래대금·시가총액은 당일 실제 값. change_rate 는 비율(0.01 = 1%)."""
    ents = [entities] if isinstance(entities, str) else list(entities)
    cols = columns or ["open", "high", "low", "close", "volume", "value", "change_rate", "market_cap"]
    q = (f'GetStockPrices([{", ".join(ents)}], columns={json.dumps(cols)}, '
         f'date_from={date_from}, date_to={date_to})')
    return ds_df(q)


_UNIVERSE = None


def universe():
    """현재 상장 종목 기본정보(GetStockSymbols 전 종목): symbol, name, market, industry_name 등.
    company_type_l1: 1 일반법인, 2 공기업(한국전력·기업은행 등 8종목), 8 ETF·ETN 등."""
    global _UNIVERSE
    if _UNIVERSE is None:
        u = ds_df("GetStockSymbols()")
        _UNIVERSE = u.rename(columns={"entity_name": "name"})[
            ["symbol", "name", "market", "company_type_l1", "industry_id", "industry_name", "date_listed",
             "fiscal_year_end", "conglomerate_id"]]
    return _UNIVERSE.copy()


def all_stocks(day, common_only=True, columns=None):
    """day(또는 그 이전 마지막 거래일)의 전 종목 하루치 시세 + 종목 정보. 한 번 호출로 4,000여 종목.
    common_only=True 이면 일반법인·공기업 보통주만(ETF·ETN·우선주 제외)."""
    day = pd.Timestamp(day).date()
    cols = columns or ["open", "high", "low", "close", "volume", "value", "change_rate", "market_cap"]
    snap = ds_df(f'GetStockPrices(columns={json.dumps(cols)}, '
                 f'date_from={day - timedelta(days=7)}, date_to={day})')
    df = snap.merge(universe().drop(columns=["date"], errors="ignore"), on="symbol", how="left")
    if common_only:
        df = df[df["company_type_l1"].isin(["1", "2"]) & df["symbol"].str.endswith("0")]
    df = df[df["market"].isin(["KOSPI", "KOSDAQ", "KONEX"])]
    if "change_rate" in df:
        df["change_pct"] = df["change_rate"] * 100
    return df.reset_index(drop=True)


def index_prices(date_from, date_to):
    """지수 11종(코스피·코스닥 계열) 일별. change_rate 는 퍼센트(1.94 = 1.94%).
    지수의 value(거래대금)는 백만원, volume(거래량)은 천주 단위로 보입니다(코스닥 2026-11-23 value 4,673,819 ≈ 4.7조)."""
    # 지수를 지정하지 않으면 최근 약 1,300행(≈118거래일)만 오고 앞 기간이 조용히 잘린다 → 140일씩 나눠 받는다
    start, end = pd.Timestamp(date_from), pd.Timestamp(date_to)
    parts = []
    while start <= end:
        stop = min(start + pd.Timedelta(days=139), end)
        parts.append(ds_df(f"GetMarketIndexes(date_from={start.date()}, date_to={stop.date()})"))
        start = stop + pd.Timedelta(days=1)
    df = pd.concat([x for x in parts if not x.empty], ignore_index=True) if parts else pd.DataFrame()
    return df.drop_duplicates(["date", "symbol"]).sort_values(["date", "symbol"]).reset_index(drop=True) if not df.empty else df


def fred(series, date_from, date_to):
    """FRED 시리즈 → 날짜 × 시리즈 넓은 표. 예: fred(["DGS10", "DEXKOUS"], "2026-10-01", "2026-11-24")"""
    ids = ", ".join(json.dumps(s) for s in series)
    df = ds_df(f"QueryFredSeriesData([{ids}], date_from={date_from}, date_to={date_to})")
    if df.empty:
        return df
    val = [c for c in df.columns if c not in ("entity_id", "date")][0]
    df["series"] = df["entity_id"].str.replace("FRED:", "", regex=False)
    wide = df.pivot_table(index="date", columns="series", values=val).sort_index()
    return wide[[s for s in series if s in wide.columns]]  # 요청한 순서대로


def krw(v, digits=1):
    """원 → '138.0조' / '5,027억'."""
    if v is None or pd.isna(v):
        return "-"
    a = abs(v)
    if a >= 1e12:
        return f"{v / 1e12:.{digits}f}조"
    if a >= 1e8:
        return f"{v / 1e8:,.0f}억"
    if a >= 1e4:
        return f"{v / 1e4:,.0f}만"
    return f"{v:,.0f}"


PRICE_NOTICE = ("※ 주가는 수정주가입니다(분할·무상증자·유상증자 소급 반영, 배당 미반영). "
                "거래량·거래대금·시가총액은 당일 실제 값이며 KRX 기준입니다(NXT 미포함).")

# 한글 차트 폰트
import matplotlib.pyplot as plt
try:
    import koreanize_matplotlib  # noqa: F401  (Colab: 위 셀에서 설치)
except ImportError:
    if sys.platform.startswith("win"):
        plt.rcParams["font.family"] = "Malgun Gothic"
    elif sys.platform == "darwin":
        plt.rcParams["font.family"] = "AppleGothic"
plt.rcParams["axes.unicode_minus"] = False

print("DeepSearch 준비 완료", "(Colab)" if IN_COLAB else "(로컬)")

DeepSearch 준비 완료 (로컬)


In [3]:
ASOF = "2026-11-24"

## 본문 받기 함수
웹 페이지를 읽는 방식이라 사이트 구조가 바뀌면 깨질 수 있습니다. 요청 사이에 0.3초씩 쉽니다.

In [4]:
import html as _html, io, re

UA = {"User-Agent": "Mozilla/5.0 (research notebook)"}

def _get(url, **kw):
    time.sleep(0.3)
    r = requests.get(url, headers=UA, timeout=30, **kw)
    r.raise_for_status()
    r.encoding = r.apparent_encoding
    return r.text

def kind_html(url):
    page = _get(url)
    doc_no = re.search(r"<option value='(\d+)\|", page).group(1)
    js = _get("https://kind.krx.co.kr/common/disclsviewer.do", params={"method": "searchContents", "docNo": doc_no})
    return _get(re.search(r"setPath\('[^']*','([^']+\.htm)'", js).group(1))

def dart_html(url):
    page = _get(url)
    a = re.search(r'viewDoc\("(\d+)",\s*"(\d+)",\s*"(\d+)",\s*"(\d+)",\s*"(\d+)",\s*"([^"]*)"', page).groups()
    return _get("https://dart.fss.or.kr/report/viewer.do",
                params=dict(zip(["rcpNo", "dcmNo", "eleId", "offset", "length", "dtd"], a)))

def body(url):
    """공시 링크 → (본문 텍스트, 표 목록)."""
    h = kind_html(url) if "kind.krx.co.kr" in url else dart_html(url)
    try:
        tables = pd.read_html(io.StringIO(h))
    except ValueError:
        tables = []
    t = re.sub(r"<style.*?</style>", " ", h, flags=re.S)
    t = re.sub(r"<br\s*/?>|</p>|</tr>", "\n", t)
    t = _html.unescape(_html.unescape(re.sub(r"<[^>]+>", " ", t)))   # 본문에 &amp;nbsp; 처럼 두 번 인코딩된 곳이 있다
    t = re.sub(r"[ \t\xa0]+", " ", t)
    return re.sub(r"\n\s*\n+", "\n", t).strip(), tables

## 1. 오리온 실적 공시 → 국가별 순매출
오리온은 월별 잠정실적 공시의 "기타 투자판단 사항"에 국가별 매출을 **서술형 표**로 적습니다. 표가 아니라 글이라 정규식으로 뽑습니다.

In [5]:
total, docs = doc_search("company", ["disclosure"], "securities.name:오리온 and title:잠정", "2026-06-01", "2026-06-30", count=10)
doc = docs.sort_values("created_at").iloc[-1]
print(doc["created_at"], doc["title"], doc["url"])
text, tables = body(doc["url"])

def country_sales(text):
    out = {}
    for block in re.finditer(r"(\d{4}년 \d+월 (?:당월|누계))\s*한국\s*중국\s*베트남\s*러시아\s*합계(.*?)(?=\d{4}년 \d+월 (?:당월|누계)|$)", text, re.S):
        m = re.search(r"순매출\**\s*([-\d,\s]+)", block.group(2))
        nums = [int(x.replace(",", "")) for x in re.findall(r"-?[\d,]+", m.group(1))][:5]
        out[block.group(1)] = nums
    return pd.DataFrame(out, index=["한국", "중국", "베트남", "러시아", "합계"])

sales = country_sales(text)
print("단위: 억원 (순매출)")
sales

2026-06-16 09:57:38 연결재무제표기준영업(잠정)실적(공정공시) https://dart.fss.or.kr/dsaf001/main.do?rcpNo=20260616800198


단위: 억원 (순매출)


,2026년 5월 당월,2026년 5월 누계
한국,1004,4849
중국,1237,6637
베트남,401,2293
러시아,369,1606
합계,3011,15385


In [6]:
print("합계 검산:", (sales.iloc[:4].sum() == sales.loc["합계"]).to_dict())
fx = re.search(r"환율효과는 다음과 같습니다.*?:\s*(.*?)\s*-\s*\d{4}년", text, re.S)
print("본문의 환율효과:", fx.group(1) if fx else "-")

합계 검산: {'2026년 5월 당월': True, '2026년 5월 누계': True}
본문의 환율효과: 중국 위안화 +13.3%, 베트남 동화 +5.8%, 러시아 루블화 +17.7%


## 2. 공매도 과열종목 지정 공시 → 대상종목·ISIN·지정일 표
거래소 공시는 정해진 서식(대상 종목·지정일·시장조치 내용)이라 표가 그대로 읽힙니다.
DeepSearch 검색 결과에서 **종목 태그가 비어 있던 공시**(주로 우선주)도 본문으로 종목을 채울 수 있습니다.

In [7]:
total, ov = doc_search("company", ["disclosure"], 'title:"공매도 과열종목 지정"', ASOF, ASOF, count=100)
ov = ov.sort_values("created_at")
print(f"{ASOF} 공매도 과열종목 지정 공시 {total}건, 그중 DeepSearch 종목 태그 없음 {(ov['symbols'] == '').sum()}건")

rows = []
for _, d in ov.iterrows():
    try:
        _, tbs = body(d["url"])
        kv = dict(zip(tbs[0][0], tbs[0][1]))
    except Exception as e:
        rows.append({"공시시각": d["created_at"], "DeepSearch 종목": d["stocks"], "대상 종목": f"(실패: {e})"})
        continue
    target = str(kv.get("대상 종목", ""))
    m = re.match(r"(.+?)\((KR\w{10})\)", target)
    rows.append({"공시시각": d["created_at"].strftime("%H:%M"), "시장": d["publisher"],
                 "DeepSearch 종목": d["stocks"] or "(태그 없음)",
                 "대상 종목": m.group(1) if m else target, "ISIN": m.group(2) if m else "",
                 "지정일(공매도 금지일)": kv.get("지정일", ""),
                 "금지 기간": re.search(r"지정일 (\S+ 간)", str(kv.get("시장조치 내용", ""))).group(1)
                 if re.search(r"지정일 (\S+ 간)", str(kv.get("시장조치 내용", ""))) else ""})
parsed = pd.DataFrame(rows)
parsed

2026-11-24 공매도 과열종목 지정 공시 6건, 그중 DeepSearch 종목 태그 없음 1건


,공시시각,시장,DeepSearch 종목,대상 종목,ISIN,지정일(공매도 금지일),금지 기간
0,20:15,유가증권시장본부,(태그 없음),케이아이이,KR7009140005,2026-11-25,1일 간
1,20:15,코스닥시장본부,큐리언트,큐리언트,KR7115180002,2026-11-25,1일 간
2,20:15,코스닥시장본부,스튜디오드래곤,스튜디오드래곤,KR7253450001,2026-11-25,1일 간
3,20:15,유가증권시장본부,한솔홈데코,한솔홈데코,KR7025750001,2026-11-25,1일 간
4,20:16,코스닥시장본부,CJ ENM,CJ ENM,KR7035760008,2026-11-25,1일 간
5,20:16,코스닥시장본부,휴림로봇,휴림로봇,KR7090710005,2026-11-25,1일 간


지정일은 공시 다음 거래일입니다. 공시가 20시께 나오고 **다음 날 하루** 공매도가 금지된다는 점이 본문으로 확인됩니다(`2-03` 트리거일 정의의 근거).

## 3. 같은 날 잠정실적 공시 → 매출·영업이익 한 표
`영업(잠정)실적(공정공시)`는 대부분 같은 서식(매출액·영업이익·순이익 × 당해실적/누계실적, 당기·전기·전년동기)이라 한꺼번에 뽑을 수 있습니다.
회사가 별도 표를 덧붙이거나 서식이 다르면 `파싱 실패`로 남깁니다.

In [8]:
EARN_DAY = "2026-11-13"     # 3분기 실적 시즌 마지막 주
total, er = doc_search("company", ["disclosure"], "title:영업(잠정)실적", EARN_DAY, EARN_DAY, count=100)
er = er[~er["title"].str.contains("정정") & (er["stocks"] != "")].drop_duplicates("stocks").head(20)
print(f"{EARN_DAY} 잠정실적 공시 {total}건 중 {len(er)}건 본문 추출")

def earnings_table(tables):
    for t in tables:
        if t.shape[1] < 8:
            continue
        cells = [str(x) for x in t.values.ravel() if "단위" in str(x)]       # 예: "단위 : 백만원, %"
        u = cells[0] if cells else ""
        unit = "백만원" if "백만원" in u else ("억원" if "억원" in u else ("원" if "원" in u else None))
        if unit is None:
            continue
        g = t[(t[1].astype(str) == "당해실적") & t[0].isin(["매출액", "영업이익", "당기순이익"])]
        if len(g):
            g = g.drop_duplicates(0).set_index(0)
            return unit, g
    return None, None

def num(x):
    try:
        return float(str(x).replace(",", ""))
    except ValueError:
        return None

out = []
for _, d in er.iterrows():
    try:
        _, tbs = body(d["url"])
        unit, g = earnings_table(tbs)
    except Exception as e:
        unit, g = None, None
    row = {"종목": d["stocks"].split(", ")[0], "제목": d["title"][:20], "단위": unit or "파싱 실패"}
    if g is not None:
        scale = {"억원": 1, "백만원": 0.01, "원": 1e-8}[unit]
        for k in ("매출액", "영업이익"):
            if k in g.index:
                cur, yoy_base = num(g.loc[k, 2]), num(g.loc[k, 6])
                row[f"{k}(억원)"] = round(cur * scale, 1) if cur is not None else None
                row[f"{k} 전년동기(억원)"] = round(yoy_base * scale, 1) if yoy_base is not None else None
                row[f"{k} YoY(%)"] = num(g.loc[k, 7])
    out.append(row)
earn = pd.DataFrame(out)
print("성공", (earn["단위"] != "파싱 실패").sum(), "/", len(earn))
earn

2026-11-13 잠정실적 공시 16건 중 12건 본문 추출


성공 12 / 12


,종목,제목,단위,매출액(억원),매출액 전년동기(억원),매출액 YoY(%),영업이익(억원),영업이익 전년동기(억원),영업이익 YoY(%)
0,콜마홀딩스,연결재무제표기준영업(잠정)실적(공정공,백만원,1345.3,1655.2,-18.70,195.2,163.3,19.50
1,티케이지휴켐스,연결재무제표기준영업(잠정)실적(공정공,백만원,4112.0,2917.4,40.90,211.3,264.0,-20.00
2,라온피플,연결재무제표기준영업(잠정)실적(공정공,백만원,293.6,198.9,47.60,-1.7,-62.2,97.30
3,옵티시스,영업(잠정)실적(공정공시),백만원,55.6,51.3,8.30,11.9,7.5,59.30
4,인스피언,영업(잠정)실적(공정공시),백만원,51.1,46.2,10.60,12.0,-2.4,NaN
5,동원모빌리티,연결재무제표기준영업(잠정)실적(공정공,백만원,1901.6,1556.1,22.20,352.0,84.5,316.50
6,현대해상,영업(잠정)실적(공정공시),백만원,43973.6,42794.8,2.80,2372.0,2454.5,-3.40
7,원익QnC,연결재무제표기준영업(잠정)실적(공정공,백만원,2836.9,2302.7,23.20,444.3,80.0,455.30
8,미래에셋생명,연결재무제표기준영업(잠정)실적(공정공,백만원,15132.2,12549.0,20.58,-430.8,617.7,NaN
9,서울보증보험,영업(잠정)실적(공정공시),백만원,6819.3,6496.2,5.00,385.1,800.6,-51.90


## 결과 읽는 법과 한계
- **정형 공시(거래소 시장조치, 잠정실적 표준 서식)는 거의 그대로 구조화**됩니다. 서술형 본문(오리온 국가별 매출)은 회사마다 문장 형식이 달라 회사별 정규식이 필요합니다.
- 본문의 숫자는 **잠정치**입니다. 확정치는 정기보고서나 `GetFinancialStatements`로 교차 확인하세요.
- KIND·DART 웹 페이지 구조에 기댄 방법입니다. 사이트가 바뀌면 `kind_html`/`dart_html`의 정규식을 고쳐야 합니다. 대량 수집은 하지 말고 필요한 공시만 받으세요.
- DeepSearch에서 공시 본문 자체는 받을 수 없습니다(`DocumentSearch`의 `content`는 제목만, `read_document.py`도 DART 프레임 구조 때문에 본문이 거의 안 나옴).